In [2]:
import pandas as pd
import json
from pathlib import Path

In [3]:
df = pd.read_csv("CTD_exposure_events.csv")

C:\Users\hp\AppData\Local\Temp\ipykernel_11620\850267291.py:1: DtypeWarning: Columns (16,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CTD_exposure_events.csv")


In [4]:
# shape and preview
print("Original shape:", df.shape)
df.head()


Original shape: (230703, 43)


,exposurestressorname,exposurestressorid,stressorsourcecategory,stressorsourcedetails,numberofstressorsamples,stressornotes,numberofreceptors,receptors,receptornotes,smokingstatus,...,phenotypename,phenotypeid,phenotypeactiondegreetype,anatomy,exposureoutcomenotes,reference,associatedstudytitles,enrollmentstartyear,enrollmentendyear,studyfactors
0,"1,1,1-trichloroethane",C024566,Commercial product|Dietary|Environmental,NaN,NaN,NaN,63.0,Children,NaN,Non-smoker,...,NaN,NaN,NaN,NaN,NaN,15743726,"The School Health Initiative: Environment, Lea...",2001.0,2001.0,NaN
1,"1,1,1-trichloroethane",C024566,Commercial product|Dietary|Environmental,NaN,NaN,NaN,63.0,Children,NaN,Non-smoker,...,NaN,NaN,NaN,NaN,NaN,15743726,"The School Health Initiative: Environment, Lea...",2001.0,2001.0,NaN
2,"1,1,1-trichloroethane",C024566,Commercial product|Dietary|Environmental,NaN,NaN,NaN,63.0,Children,NaN,Non-smoker,...,NaN,NaN,NaN,NaN,NaN,15743726,"The School Health Initiative: Environment, Lea...",2001.0,2001.0,NaN
3,"1,1,1-trichloroethane",C024566,Commercial product|Dietary|Environmental,NaN,NaN,NaN,63.0,Children,NaN,Non-smoker,...,NaN,NaN,NaN,NaN,NaN,15743726,"The School Health Initiative: Environment, Lea...",2001.0,2001.0,NaN
4,"1,1,1-trichloroethane",C024566,Commercial product|Dietary|Environmental,NaN,NaN,NaN,63.0,Children,NaN,Non-smoker,...,NaN,NaN,NaN,NaN,NaN,15743726,"The School Health Initiative: Environment, Lea...",2001.0,2001.0,NaN


In [5]:
# Drop rows where chemical is missing
df = df.dropna(subset=["exposurestressorname"])

# Drop rows where both disease and phenotype outcome are missing
df = df[~(df["diseasename"].isna() & df["phenotypename"].isna())]

print("After dropping invalid rows:", df.shape)


After dropping invalid rows: (11538, 43)


In [6]:
# Normalize the 'outcomerelationship' field
def normalize_relation(value):
    if pd.isna(value):
        return "linked_to"
    value = value.strip().lower()
    if "caus" in value:
        return "causes"
    elif "associat" in value:
        return "associated_with"
    else:
        return "linked_to"

df["Relation"] = df["outcomerelationship"].apply(normalize_relation)
df[["outcomerelationship", "Relation"]].drop_duplicates().head()


,outcomerelationship,Relation
52,no correlation,linked_to
57,prediction/hypothesis,linked_to
84,positive correlation,linked_to
1553,negative correlation,linked_to
176028,NaN,linked_to


In [7]:
# Combine 'diseasename' and 'phenotypename' into a single target field
def resolve_tail(row):
    return row["diseasename"] if pd.notna(row["diseasename"]) else row["phenotypename"]

df["Outcome"] = df.apply(resolve_tail, axis=1)

# Drop rows where outcome == stressor name (self-loop)
df = df[df["exposurestressorname"].str.strip().str.lower() != df["Outcome"].str.strip().str.lower()]

print("After resolving outcome and removing self-loops:", df.shape)
df[["exposurestressorname", "Relation", "Outcome"]].head()


After resolving outcome and removing self-loops: (11538, 45)


,exposurestressorname,Relation,Outcome
52,"1,1,1-trichloroethane",linked_to,Amyotrophic Lateral Sclerosis
57,"1,1,2,2-tetrachloroethane",linked_to,Neoplasms
76,"1,1,2,2-tetrachloroethane",linked_to,Amyotrophic Lateral Sclerosis
84,"1,12-benzoperylene",linked_to,cholesterol homeostasis
138,"1,2,3,4,6,7,8-heptachlorodibenzodioxin",linked_to,regulation of blood pressure


In [8]:
# Keep only necessary columns
df = df[["exposurestressorname", "Relation", "Outcome", "reference"]]

# Drop duplicate triplets
df = df.drop_duplicates(subset=["exposurestressorname", "Relation", "Outcome"])

print("After deduplication:", df.shape)


After deduplication: (5820, 4)


In [9]:
triplets = []

for _, row in df.iterrows():
    triplets.append({
        "head": row["exposurestressorname"].strip(),
        "relation": row["Relation"],
        "tail": row["Outcome"].strip(),
        "source": "CTD_exposure_events",
        "pubmed_ids": [row["reference"]] if pd.notna(row["reference"]) else []
    })

print("Sample triplet:")
triplets[0]


Sample triplet:


{'head': '1,1,1-trichloroethane',
 'relation': 'linked_to',
 'tail': 'Amyotrophic Lateral Sclerosis',
 'source': 'CTD_exposure_events',
 'pubmed_ids': [25544309]}

In [ ]:
output_path = Path("triplets_exposure_cleaned.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(triplets, f, indent=2)

print(f"Saved {len(triplets)} cleaned triplets to {output_path}")


✅ Saved 5820 cleaned triplets to triplets_exposure_cleaned.json
